In [1]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import spacy
import optuna
import mlflow
import mlflow.sklearn

# MLflow / DagShub tracking setup
os.environ["MLFLOW_TRACKING_USERNAME"] = "Roy7721"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "bc912b7d58bd2aca05abdc192e010414493a3886"
mlflow.set_tracking_uri("https://dagshub.com/Roy7721/yt_comment_analysis.mlflow")
mlflow.set_experiment("LogisticRegression HP Tuning with Custom Features")

C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/84a8f7afc8434d5398ba8feeab10e4f4', creation_time=1784660079577, effective_trace_archival_retention=None, experiment_id='14', last_update_time=1784660079577, lifecycle_stage='active', name='LogisticRegression HP Tuning with Custom Features', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [2]:
# Load dataset
dataset = pd.read_csv('dataset.csv')

# Drop rows with NaN values in 'clean_comment'
cleaned_dataset = dataset.dropna()

In [3]:
# Separate features and target
X_cleaned = cleaned_dataset['clean_comment']
y_cleaned = cleaned_dataset['category']

# Split the cleaned data into train and test sets (80-20 split)
X_train_cleaned, X_test_cleaned, y_train_cleaned, y_test_cleaned = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42)


In [4]:
# Load spacy language model for POS tagging
nlp = spacy.load('en_core_web_sm')

In [5]:
# All 17 universal POS tags, in a FIXED order.
# Looping over this (instead of set(pos_tags)) guarantees every comment produces the
# same 17 POS columns in the same order -> train, test, and future data always line up.
UNIVERSAL_POS = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM',
                 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']

# Function to extract custom features
def extract_custom_features(text):
    doc = nlp(text)
    word_list = [token.text for token in doc]

    # 1. Comment Length (number of characters)
    comment_length = len(text)

    # 2. Word Count
    word_count = len(word_list)

    # 3. Average Word Length
    avg_word_length = sum(len(word) for word in word_list) / word_count if word_count > 0 else 0

    # 4. Unique Word Count
    unique_word_count = len(set(word_list))

    # 5. Lexical Diversity
    lexical_diversity = unique_word_count / word_count if word_count > 0 else 0

    # 6. Count of POS Tags
    pos_count = len([token.pos_ for token in doc])

    # 7. Proportion of POS Tags - loop over the FIXED list (not set()) so every comment
    #    yields all 17 columns in the same order; tags not present come out as 0.
    pos_tags = [token.pos_ for token in doc]
    if word_count > 0:
        pos_proportion = {tag: pos_tags.count(tag) / word_count for tag in UNIVERSAL_POS}
    else:
        pos_proportion = {tag: 0 for tag in UNIVERSAL_POS}

    return {
        'comment_length': comment_length,
        'word_count': word_count,
        'avg_word_length': avg_word_length,
        'unique_word_count': unique_word_count,
        'lexical_diversity': lexical_diversity,
        'pos_count': pos_count,
        **pos_proportion  # Flattening the POS proportions
    }

In [ ]:
# Apply the custom feature extraction
train_custom_features = pd.DataFrame([extract_custom_features(text) for text in X_train_cleaned])
test_custom_features = pd.DataFrame([extract_custom_features(text) for text in X_test_cleaned])

In [ ]:
train_custom_features.head()

In [ ]:
# Replace NaN values in POS tag proportions with 0
train_custom_features.fillna(0, inplace=True)
test_custom_features.fillna(0, inplace=True)

In [ ]:
test_custom_features.isnull().sum()

In [ ]:
# Apply TfidfVectorizer with bigram setting and max_features=10000
ngram_range = (1, 2)
max_features = 10000
tfidf = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_tfidf = tfidf.fit_transform(X_train_cleaned)
X_test_tfidf = tfidf.transform(X_test_cleaned)

In [ ]:
# Keep TF-IDF SPARSE - no .toarray() here.
# The TF-IDF matrix is ~99.8% zeros; densifying it would cost ~2.35 GB and force the
# solver to multiply by zero hundreds of millions of times per iteration.
#
# All we need before stacking is that the small 23-column custom-feature block has the
# same columns in the same order for train and test.
test_custom_features = test_custom_features.reindex(
    columns=train_custom_features.columns, fill_value=0
)

print('custom feature columns aligned:', list(train_custom_features.columns) == list(test_custom_features.columns))

In [ ]:
from scipy.sparse import hstack, csr_matrix

# Combine [sparse TF-IDF | small dense custom-feature block] WITHOUT densifying anything.
# Column order is guaranteed: TF-IDF columns come from the fitted vocabulary (identical for
# train/test), and the custom block was aligned in the previous cell. Sparse matrices carry
# no column names, so the old "feature names must be in the same order" error cannot occur.
X_train_combined = hstack([X_train_tfidf, csr_matrix(train_custom_features.values)]).tocsr()
X_test_combined  = hstack([X_test_tfidf,  csr_matrix(test_custom_features.values)]).tocsr()

dense_mb = X_train_combined.shape[0] * X_train_combined.shape[1] * 8 / 1e6
sparse_mb = (X_train_combined.data.nbytes + X_train_combined.indices.nbytes + X_train_combined.indptr.nbytes) / 1e6
print('train shape :', X_train_combined.shape)
print('test  shape :', X_test_combined.shape)
print(f'sparse size : {sparse_mb:.1f} MB   (dense would be ~{dense_mb:.0f} MB)')
print(f'density     : {X_train_combined.nnz / (X_train_combined.shape[0] * X_train_combined.shape[1]) * 100:.2f}%  non-zero')

In [ ]:
X_train_combined

In [ ]:
X_train = X_train_combined
X_test = X_test_combined
y_train = y_train_cleaned
y_test = y_test_cleaned

In [ ]:
# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test, params, trial_number, log_model=False):
    with mlflow.start_run():
        # Log model type and trial number
        mlflow.set_tag("mlflow.runName", f"Trial_{trial_number}_{model_name}_")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("vectorizer_type", "TfidfVectorizer")
        mlflow.log_param("ngram_range", str(ngram_range))
        mlflow.log_param("vectorizer_max_features", max_features)
        mlflow.log_param("scaler", "MaxAbsScaler")

        # Log hyperparameters
        for key, value in params.items():
            mlflow.log_param(key, value)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Only log the model artifact for the best run to stay within DagShub free tier limits
        if log_model:
            # Persist locally first so a long train is never lost to a logging failure
            joblib.dump(model, f"{model_name}_best_model.joblib")
            try:
                mlflow.sklearn.log_model(
                    model, f"{model_name}_model",
                    serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
                )
            except Exception as e:
                print(f"[warn] model artifact logging failed (metrics/params + joblib file are safe): {e}")

        return accuracy

In [ ]:
# Step 6: Optuna objective function for Logistic Regression
def objective_logreg(trial):
    # Hyperparameter space to explore
    C = trial.suggest_float('C', 1e-4, 10.0, log=True)  # inverse regularization strength
    # sklearn >= 1.8 deprecated `penalty` in favour of a continuous `l1_ratio` dial:
    #   0.0 = pure L2   |   0.5 = elasticnet blend   |   1.0 = pure L1
    l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
    # Class imbalance handling - let Optuna decide whether balancing helps
    class_weight = trial.suggest_categorical('class_weight', [None, 'balanced'])

    # Log trial parameters
    params = {
        'C': C,
        'l1_ratio': l1_ratio,
        'class_weight': class_weight,
        'solver': 'saga',   # saga supports the full l1/l2/elasticnet range
        'max_iter': 300
    }

    # MaxAbsScaler -> LogisticRegression pipeline.
    # MaxAbsScaler is sparse-safe (unlike StandardScaler, which would destroy sparsity).
    # No `penalty` argument - l1_ratio fully controls the regularization type.
    model = Pipeline([
        ('scaler', MaxAbsScaler()),
        ('clf', LogisticRegression(C=C, l1_ratio=l1_ratio, class_weight=class_weight,
                                   solver='saga', max_iter=300, random_state=42))
    ])

    # Log each trial as a separate run in MLflow
    accuracy = log_mlflow("LogisticRegression", model, X_train, X_test, y_train, y_test, params, trial.number)

    return accuracy

In [ ]:
# Step 7: Run Optuna for Logistic Regression, log the best model, and plot the importance of each parameter
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_logreg, n_trials=50)  # 50 trials

    # Get the best parameters
    best_params = study.best_params
    best_model = Pipeline([
        ('scaler', MaxAbsScaler()),
        ('clf', LogisticRegression(C=best_params['C'],
                                   l1_ratio=best_params['l1_ratio'],
                                   class_weight=best_params['class_weight'],
                                   solver='saga', max_iter=300, random_state=42))
    ])

    # Log the best model with MLflow (only this run uploads the model artifact)
    log_mlflow("LogisticRegression", best_model, X_train, X_test, y_train, y_test, best_params, "Best", log_model=True)

    # Plots are optional - never let a missing optional dependency kill the return value
    try:
        optuna.visualization.plot_param_importances(study).show()
        optuna.visualization.plot_optimization_history(study).show()
    except Exception as e:
        print(f"[warn] optuna plots skipped: {e}")

    print(f"\nBest accuracy: {study.best_value:.4f}")
    print(f"Best params  : {best_params}")

    # Return the best model so it can be inspected outside this function
    return best_model

In [ ]:
# Run the experiment for Logistic Regression
best_model = run_optuna_experiment()

In [ ]:
best_model